# 文脈化ベクトルをクラスタリングして語義を自動で分ける（語義推定）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_lexical_semantics/word_sense_induction.ipynb)

NLP クックブック 19-06 ／ 語の意味と意味の変化

辞書の語義に頼らずに、コーパス中の用例を「同じ意味で使われているもの同士」にグループ分けしたい。新語や専門用語の意味の分析に使える。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import torch
from transformers import AutoModel, AutoTokenizer

name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name).eval()

def target_vector(sentence, target):
    """文中の target に当たるトークンの文脈化ベクトル (複数トークンなら平均) を返す"""
    enc = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    pieces = tokenizer.tokenize(target)
    start = next(i for i in range(len(tokens)) if tokens[i:i + len(pieces)] == pieces)
    with torch.no_grad():
        hidden = model(**enc, output_hidden_states=True).hidden_states
    layers = torch.stack(hidden[-4:]).mean(0)[0]              # 最後の 4 層の平均
    return layers[start:start + len(pieces)].mean(0)

sentences = [
    "タクシーのドライバーに行き先を伝えた。", "長距離トラックのドライバーは睡眠不足になりやすい。", "彼は F1 のドライバーとして活躍した。", "バスのドライバーが不足している。",
    "プリンタのドライバーをインストールし直した。", "グラフィックボードのドライバーを最新版に更新する。", "古い OS 用のドライバーが見つからない。", "ドライバーの不具合で画面が映らない。",
    "ネジを締めるためにプラスのドライバーを使う。", "工具箱からドライバーとペンチを取り出した。", "精密ドライバーで眼鏡のネジを直した。", "電動ドライバーがあると家具の組み立てが楽だ。",
]
X = torch.stack([target_vector(s, "ドライバー") for s in sentences])
X = torch.nn.functional.normalize(X, dim=-1).numpy()
print(X.shape)

(12, 768)


In [3]:
from sklearn.cluster import KMeans

labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
for c in sorted(set(labels)):
    print(c, [s for s, l in zip(sentences, labels) if l == c])

0 ['プリンタのドライバーをインストールし直した。', 'グラフィックボードのドライバーを最新版に更新する。', '古い OS 用のドライバーが見つからない。', 'ドライバーの不具合で画面が映らない。']
1 ['ネジを締めるためにプラスのドライバーを使う。', '工具箱からドライバーとペンチを取り出した。', '精密ドライバーで眼鏡のネジを直した。', '電動ドライバーがあると家具の組み立てが楽だ。']
2 ['タクシーのドライバーに行き先を伝えた。', '長距離トラックのドライバーは睡眠不足になりやすい。', '彼は F1 のドライバーとして活躍した。', 'バスのドライバーが不足している。']


In [4]:
# 用例同士の類似度を見る: 同じ語義の用例は近く、違う語義の用例は遠い
import pandas as pd

sim = pd.DataFrame(X @ X.T, index=[s[:10] for s in sentences]).round(2)
sim.iloc[[0, 1, 4, 5, 8, 9], [0, 1, 4, 5, 8, 9]]

,0,1,4,5,8,9
タクシーのドライバー,1.00,0.80,0.75,0.68,0.72,0.73
長距離トラックのドラ,0.80,1.00,0.77,0.78,0.80,0.77
プリンタのドライバー,0.75,0.77,1.00,0.93,0.85,0.83
グラフィックボードの,0.68,0.78,0.93,1.00,0.83,0.79
ネジを締めるためにプ,0.72,0.80,0.85,0.83,1.00,0.90
工具箱からドライバー,0.73,0.77,0.83,0.79,0.90,1.00


- 同じ「ドライバー」でも、運転手・ソフトウェア・工具で文脈化ベクトルが分かれる。これを利用して、語義の一覧を与えずに用例を分類するのが語義推定 (word sense induction)。
- クラスタ数は自分で決める必要がある。語義の数は用途によって変わるので、唯一の正解はない。
- 話題がはっきり違う語義は分かれやすい。「高い」（値段・背・声）のように同じ構文で使われる語義は、ベクトルが近くなって分かれにくい。
- 時期の違うコーパスで同じ語のベクトルの分布を比べると、意味変化の検出（「やばい」の意味の広がりなど）に応用できる。